# Data Science with NumPy and pandas
* welcome to the "specialization" part of this course - lessons 01-18 taught you python itself, this and the following lessons build on top of that to cover real, popular application areas: data science, machine learning, AI/LLMs, web development and automation
* data science is all about working with LARGE amounts of numeric/tabular data efficiently - plain python lists and loops get slow and clunky fast once you have thousands or millions of values
* two libraries dominate the python data science world: **NumPy** (fast numeric arrays) and **pandas** (spreadsheet-like tables built on top of NumPy)
* both are third-party packages (`pip install numpy pandas`), not part of the standard library - but they are SO widely used that "python for data" basically means "python + numpy + pandas" in practice

## why not just use plain python lists?
* a plain python list can hold anything, and every operation on it loops in slow, interpreted python bytecode (remember lesson 16: python is interpreted, dynamic typing has overhead)
* a NumPy array holds only ONE fixed datatype and stores its values in one tight, contiguous block of memory (much like a C array) - operations on it run as compiled, vectorized C code under the hood, often 10-100x faster than an equivalent python for-loop
* lets prove the vectorization idea with a tiny, fast comparison (kept small so it runs quickly, this is illustrative, not a rigorous benchmark like we discussed in lesson 16)

In [1]:
import numpy as np  # "np" is the universal, everyone-uses-it alias for numpy
import time

size = 500_000

# plain python: a list of numbers, doubled with a for-loop (classic, slow-ish approach)
python_list = list(range(size))
start = time.perf_counter()
doubled_list = [n * 2 for n in python_list]  # still a python-level loop under the hood, even as a comprehension
python_time = time.perf_counter() - start

# numpy: the exact same "double every number" operation, but VECTORIZED - no explicit python loop at all
numpy_array = np.arange(size)  # np.arange is numpy's version of range(), but returns an array
start = time.perf_counter()
doubled_array = numpy_array * 2  # this single expression doubles EVERY element at once, in compiled C code
numpy_time = time.perf_counter() - start

print(f"plain python list comprehension: {python_time:.4f}s")
print(f"numpy vectorized multiplication: {numpy_time:.4f}s")
print(f"numpy was roughly {python_time / numpy_time:.1f}x faster on this machine (dont read too much into the exact ratio, see lesson 16)")

plain python list comprehension: 0.0188s
numpy vectorized multiplication: 0.0021s
numpy was roughly 8.9x faster on this machine (dont read too much into the exact ratio, see lesson 16)


## numpy arrays: the basics
* `np.array(...)` builds an array from a python list (or list of lists for multiple dimensions)
* every numpy array has a `.shape` (its dimensions), a `.dtype` (the single shared datatype of all its elements) and supports "vectorized" math directly, no loops needed

In [2]:
# a 1-dimensional array, built from a plain python list
scores = np.array([85, 92, 78, 90, 88])
print(scores)
print("shape:", scores.shape)   # (5,) - one dimension, 5 elements
print("dtype:", scores.dtype)   # int64 (or similar) - numpy picked ONE shared datatype automatically

# vectorized math - no for-loop needed, these operate on EVERY element at once
print("plus 5 bonus points:", scores + 5)
print("as a percentage:", scores / 100)
print("who passed (>=80)?:", scores >= 80)  # this returns a BOOLEAN array, one True/False per element

# aggregation functions - numpy comes with these built in, no manual loop-and-sum needed
print("mean:", scores.mean())
print("max:", scores.max(), "min:", scores.min())
print("standard deviation:", round(scores.std(), 2))

# a 2-dimensional array (like a small grid/matrix) - built from a list of lists
grid = np.array([[1, 2, 3], [4, 5, 6]])
print("2d array:\n", grid)
print("shape:", grid.shape)  # (2, 3) - 2 rows, 3 columns
print("sum of everything:", grid.sum())
print("sum per column:", grid.sum(axis=0))  # axis=0 sums DOWN each column
print("sum per row:", grid.sum(axis=1))     # axis=1 sums ACROSS each row

[85 92 78 90 88]
shape: (5,)
dtype: int64
plus 5 bonus points: [90 97 83 95 93]
as a percentage: [0.85 0.92 0.78 0.9  0.88]
who passed (>=80)?: [ True  True False  True  True]
mean: 86.6
max: 92 min: 78
standard deviation: 4.88
2d array:
 [[1 2 3]
 [4 5 6]]
shape: (2, 3)
sum of everything: 21
sum per column: [5 7 9]
sum per row: [ 6 15]


## boolean indexing / filtering - numpy's superpower
* we can use a boolean array (like `scores >= 80` above) directly as an "index" to filter an array - no explicit loop or `if` needed
* this pattern ("boolean masking") is used constantly in real numpy/pandas code

In [3]:
scores = np.array([85, 92, 78, 90, 88])

passing_mask = scores >= 80          # a boolean array: [True, True, False, True, True]
print("mask:", passing_mask)
print("only the passing scores:", scores[passing_mask])  # numpy keeps only the elements where the mask is True

# this is often written as one compact line, filter and all
print("failing scores in one line:", scores[scores < 80])

# we can also just count how many passed, since True acts like 1 and False like 0 when summed
print("how many passed:", passing_mask.sum())

mask: [ True  True False  True  True]
only the passing scores: [85 92 90 88]
failing scores in one line: [78]
how many passed: 4


## pandas: Series and DataFrame
* pandas builds two main structures on top of numpy: a `Series` (a single labeled 1D column) and a `DataFrame` (a full labeled table, like a spreadsheet or a SQL table - literally a collection of Series sharing the same row labels/"index")
* if you know spreadsheets or SQL, most of pandas will feel very familiar, just with python syntax instead of formulas/SQL clauses

In [4]:
import pandas as pd  # "pd" is the universal alias for pandas, just like "np" for numpy

# a Series - one labeled column of data. by default the labels (the "index") are just 0, 1, 2, ...
grades = pd.Series([85, 92, 78, 90, 88], name="score")
print(grades)
print("mean:", grades.mean())  # Series supports the same aggregation methods numpy arrays do

# a DataFrame - built here from a python dict: each key becomes a column name, each value-list becomes that column's data
students = pd.DataFrame({
    "name": ["Alex", "Mia", "Sam", "Jo", "Lee"],
    "score": [85, 92, 78, 90, 88],
    "subject": ["math", "math", "science", "science", "math"],
})
print(students)
print()
print("shape:", students.shape)     # (rows, columns)
print("column names:", list(students.columns))
print("data types per column:\n", students.dtypes)

0    85
1    92
2    78
3    90
4    88
Name: score, dtype: int64
mean: 86.6
   name  score  subject
0  Alex     85     math
1   Mia     92     math
2   Sam     78  science
3    Jo     90  science
4   Lee     88     math

shape: (5, 3)
column names: ['name', 'score', 'subject']
data types per column:
 name         str
score      int64
subject      str
dtype: object


## selecting, filtering and creating columns
* `df["column"]` selects one column (returns a Series), `df[["a", "b"]]` selects several columns (returns a DataFrame)
* `df[boolean_condition]` filters ROWS - the exact same boolean-masking idea as numpy, just applied to a whole table
* `.loc[row_condition, "column"]` is the more explicit/flexible way to select rows AND columns together, and is also how you should assign new values back (avoids a common pandas warning about "chained assignment")

In [5]:
# selecting one column - returns a Series
print(students["score"])
print()

# selecting several columns - returns a smaller DataFrame
print(students[["name", "score"]])
print()

# filtering rows - only students who scored 85 or higher
high_scorers = students[students["score"] >= 85]
print(high_scorers)
print()

# combining conditions - use & (and) / | (or), and wrap each condition in parentheses (a common pandas gotcha!)
math_high_scorers = students[(students["subject"] == "math") & (students["score"] >= 85)]
print(math_high_scorers)
print()

# creating a brand new column - just assign to a column name that doesnt exist yet, pandas creates it
students["passed"] = students["score"] >= 80   # a whole new boolean column, computed from the score column
print(students)

0    85
1    92
2    78
3    90
4    88
Name: score, dtype: int64

   name  score
0  Alex     85
1   Mia     92
2   Sam     78
3    Jo     90
4   Lee     88

   name  score  subject
0  Alex     85     math
1   Mia     92     math
3    Jo     90  science
4   Lee     88     math

   name  score subject
0  Alex     85    math
1   Mia     92    math
4   Lee     88    math

   name  score  subject  passed
0  Alex     85     math    True
1   Mia     92     math    True
2   Sam     78  science   False
3    Jo     90  science    True
4   Lee     88     math    True


## grouping and aggregating - the "group by category and summarize" pattern
* `.groupby("column")` groups rows sharing the same value in that column, then we can call an aggregation like `.mean()`, `.sum()`, `.count()` on the result
* this is the exact same idea as SQL's `GROUP BY` (lesson 13) or a spreadsheet pivot table - and its one of the single most useful pandas operations in real data work

In [6]:
# average score per subject
average_by_subject = students.groupby("subject")["score"].mean()
print(average_by_subject)
print()

# multiple aggregations at once with .agg()
summary = students.groupby("subject")["score"].agg(["mean", "min", "max", "count"])
print(summary)

subject
math       88.333333
science    84.000000
Name: score, dtype: float64

              mean  min  max  count
subject                            
math     88.333333   85   92      3
science  84.000000   78   90      2


## handling missing data
* real data is messy - values are often missing, represented in pandas as `NaN` ("Not a Number", from numpy)
* `.isna()` finds missing values, `.dropna()` removes rows with any missing value, `.fillna(...)` replaces missing values with something sensible (a default, a mean, ...)

In [7]:
messy_data = pd.DataFrame({
    "name": ["Alex", "Mia", "Sam", "Jo"],
    "score": [85, None, 78, None],  # None becomes NaN once its inside a numeric pandas column
})
print(messy_data)
print()
print("which scores are missing?\n", messy_data["score"].isna())
print()

# option 1: drop rows with any missing value entirely
print("after dropna():\n", messy_data.dropna())
print()

# option 2: fill missing values with something reasonable, e.g. the column average
mean_score = messy_data["score"].mean()  # pandas automatically ignores NaN when computing mean()
filled_data = messy_data.fillna({"score": mean_score})
print(f"after fillna() with the mean ({mean_score}):\n", filled_data)

   name  score
0  Alex   85.0
1   Mia    NaN
2   Sam   78.0
3    Jo    NaN

which scores are missing?
 0    False
1     True
2    False
3     True
Name: score, dtype: bool

after dropna():
    name  score
0  Alex   85.0
2   Sam   78.0

after fillna() with the mean (81.5):
    name  score
0  Alex   85.0
1   Mia   81.5
2   Sam   78.0
3    Jo   81.5


## reading and writing real files
* pandas can read/write almost any tabular file format directly - `read_csv`/`to_csv`, `read_json`/`to_json`, and many more (excel, sql, parquet, ...)
* this connects directly back to lesson 05 (files/csv/json) and lesson 13 (databases) - pandas is often the tool you reach for once "a few rows in a csv" grows into "thousands of rows I need to analyze"

In [8]:
import os

# writing our students DataFrame to an actual csv file
students.to_csv("demo_students.csv", index=False)  # index=False means dont write the 0,1,2,... row labels as their own column

# reading it back - this is the pandas equivalent of the csv.DictReader loop we wrote by hand in lesson 05
reloaded = pd.read_csv("demo_students.csv")
print(reloaded)

os.remove("demo_students.csv")  # cleaning up the demo file, we dont want to leave it in the repo
print("\ndemo_students.csv removed again")

   name  score  subject  passed
0  Alex     85     math    True
1   Mia     92     math    True
2   Sam     78  science   False
3    Jo     90  science    True
4   Lee     88     math    True

demo_students.csv removed again


## merging two DataFrames
* `pd.merge()` combines two DataFrames based on a shared column - the exact same idea as a SQL `JOIN` (lesson 13)

In [9]:
subjects_info = pd.DataFrame({
    "subject": ["math", "science"],
    "teacher": ["Mr. Chen", "Ms. Patel"],
})

# merge students with subjects_info, matching rows where "subject" is the same in both tables
merged = pd.merge(students, subjects_info, on="subject")
print(merged[["name", "subject", "score", "teacher"]])

# this is conceptually identical to: SELECT ... FROM students JOIN subjects_info ON students.subject = subjects_info.subject

   name  subject  score    teacher
0  Alex     math     85   Mr. Chen
1   Mia     math     92   Mr. Chen
2   Sam  science     78  Ms. Patel
3    Jo  science     90  Ms. Patel
4   Lee     math     88   Mr. Chen


## quick comparison to other languages/tools
* **R** was built from the ground up FOR statistics/data analysis - its `data.frame` (and the tidyverse's `tibble`) is the direct ancestor/cousin of pandas' `DataFrame`, and R still has a strong niche in academic statistics, but pandas + python has become the more general, popular choice in industry
* **SQL** (lesson 13) and pandas overlap a lot conceptually: `SELECT`≈column selection, `WHERE`≈boolean filtering, `GROUP BY`≈`.groupby()`, `JOIN`≈`pd.merge()` - pandas is essentially "SQL you can mix freely with normal python code and plotting"
* **Excel/Google Sheets**: a pandas DataFrame is a spreadsheet you control with code instead of clicking - this makes repeating an analysis on new data trivial (re-run the script) versus repeating manual spreadsheet clicks
* **JavaScript** has no true pandas equivalent in the standard ecosystem - libraries like `danfo.js` exist but are far less mature/popular; heavy data work in a JS-based stack is often still done by calling out to python or a real database instead
* **Java/C#** data work typically leans on Apache Spark (which has python, Java, Scala AND R APIs) for BIG data, or plain SQL + ORMs for smaller relational data - neither language has an in-language "pandas equivalent" nearly as popular

## Exercises
1. create a numpy array of the numbers 1 to 20 (`np.arange`), then use boolean masking to select only the numbers divisible by 3
2. create a small DataFrame of at least 4 products with columns `name`, `price` and `category`, then filter it to only products in one category costing more than a value you choose
3. using the `students` DataFrame from this lesson (or a similar one you build yourself), group by `subject` and find which subject has the higher average score
4. create a DataFrame with a `price` column that has one missing value (`None`), then fill the missing value with the column's mean using `.fillna()`

In [10]:
# TODO: solve exercise 1 here (numpy array 1-20, divisible by 3)


# TODO: solve exercise 2 here (products DataFrame, filter by category + price)


# TODO: solve exercise 3 here (group students by subject, compare average scores)


# TODO: solve exercise 4 here (DataFrame with a missing price, fillna with the mean)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
# sample solution 1
numbers = np.arange(1, 21)  # 1 to 20 inclusive
divisible_by_three = numbers[numbers % 3 == 0]
print(divisible_by_three)

# sample solution 2
products = pd.DataFrame({
    "name": ["Laptop", "Mouse", "Desk", "Monitor"],
    "price": [999.0, 25.0, 150.0, 249.0],
    "category": ["electronics", "electronics", "furniture", "electronics"],
})
expensive_electronics = products[(products["category"] == "electronics") & (products["price"] > 100)]
print(expensive_electronics)

# sample solution 3
students_for_ex3 = pd.DataFrame({
    "name": ["Alex", "Mia", "Sam", "Jo", "Lee"],
    "score": [85, 92, 78, 90, 88],
    "subject": ["math", "math", "science", "science", "math"],
})
averages = students_for_ex3.groupby("subject")["score"].mean()
print(averages)
print("subject with the higher average:", averages.idxmax())  # idxmax() returns the INDEX LABEL of the biggest value

# sample solution 4
products_with_gap = pd.DataFrame({"name": ["A", "B", "C"], "price": [10.0, None, 30.0]})
mean_price = products_with_gap["price"].mean()
products_filled = products_with_gap.fillna({"price": mean_price})
print(products_filled)

[ 3  6  9 12 15 18]
      name  price     category
0   Laptop  999.0  electronics
3  Monitor  249.0  electronics
subject
math       88.333333
science    84.000000
Name: score, dtype: float64
subject with the higher average: math
  name  price
0    A   10.0
1    B   20.0
2    C   30.0


* congratulation you finished this lesson

## what we learned
* why numpy arrays are much faster than plain python lists for numeric work (vectorization, one shared dtype, contiguous memory)
* numpy basics: `.shape`, `.dtype`, vectorized math, aggregations (`.mean()`, `.sum()`, ...), and boolean-mask filtering
* pandas `Series` (one labeled column) and `DataFrame` (a full labeled table)
* selecting columns, filtering rows with boolean conditions, and creating new columns
* `.groupby()` + aggregation, the same idea as SQL's `GROUP BY`
* handling missing data with `.isna()`, `.dropna()`, `.fillna()`
* reading/writing csv files with `read_csv`/`to_csv`, and combining tables with `pd.merge()` (like a SQL `JOIN`)